# 03 · Caching: exact, semantic, prompt and KV caches

Build an exact-match cache with the right key, a semantic cache with a threshold you *measured* (and watch it serve
the wrong answer), invalidate with TTLs and versioned keys, and do the arithmetic for provider prompt caching.

**Time:** ~30 min · **Runs:** offline on CPU in ~1 min (loads a 90 MB embedding model) · **Needs:** [embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb)

## Why this matters in interviews

- The cheapest LLM call is the one you never make: a cache hit costs milliseconds and nothing. Caching tops every
  "cut the bill" and "cut the latency" list.
- There are at least four different caches (response, semantic, provider prompt cache, KV cache) doing different
  jobs. Naming them and what each keys on is a senior signal.
- The semantic cache is the one cache that can **confidently serve the wrong answer**. Interviewers want you to say
  that first, then say how you would measure and bound the error.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `po09` | What are the caching layers in a GenAI application? |
| `in39` | What is a semantic cache and what makes it risky? |
| `po10` | What is semantic caching and what threshold would you set? |
| `in40` | Design a cache for an LLM that gets the same question a hundred different ways. |
| `in42` | How do you keep a cache from serving stale answers when the underlying data changes? |
| `in41` | What is cache penetration, and how is it different from a stampede? |
| `in03` | What is prompt caching and how is it different from a KV cache? |
| `pd19` | How do you keep prompt caching working as your system evolves? |
| `pd18` | What is the difference between a cache hit rate and a cost saving? |
| `sd20` | Design the caching architecture for a high-traffic LLM product. |

## Setup

Offline, each request is made to take 0.3 s (`OFFLINE.latency`) so a cache hit is visibly faster, and
`OFFLINE.request_count` tells us how many requests really reached the "provider".

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import asyncio
import hashlib
import json
import random
import re
import time
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import tiktoken
from llm_setup import OFFLINE, embed

SYSTEM = """You are Acme's customer assistant. Answer in one sentence using only the FAQ below.
FAQ:
Our opening hours are 9am to 6pm, Monday to Saturday.
Refunds are processed within 5 business days of approval.
You can reset your password from the sign-in page by choosing Forgot password.
We ship to Canada, the US and Mexico, and shipping is free on orders over $50.
Students get a 20% discount with a valid student email.
You can cancel your subscription at any time under Settings, then Billing.
To contact support, use the in-app chat or email support@acme.example.
We accept Visa, Mastercard, PayPal and Apple Pay."""

client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=5)  # warm-up
if OFFLINE:
    OFFLINE.latency = 0.3          # offline only: each request takes 0.3 s, like a short real call
def served():
    return OFFLINE.request_count if OFFLINE else None
print(f"provider={PROVIDER}, model={MODEL}")

## 1. Four caches, four different jobs

| Layer | Keyed on | What a hit saves | Can it be wrong? | Typical hit rate |
|---|---|---|---|---|
| **Exact response cache** (yours, Redis) | hash of normalised prompt + model + params + prompt version + tenant | the whole call: latency and cost | no, if the key is complete | low on free text, high on templated or programmatic calls |
| **Semantic cache** (yours, vector index) | embedding of the query, plus the same metadata as filters | the whole call | **yes**: similar is not the same | roughly 20-50% on FAQ-shaped traffic |
| **Provider prompt cache** (OpenAI, Anthropic, Gemini) | the exact token prefix of the prompt | prefill: a discount on cached input tokens and a lower time-to-first-token | no | whatever your prompt layout allows |
| **KV / prefix cache** (vLLM, SGLang, your GPUs) | token blocks of shared prefixes | prefill compute on your own hardware | no | same as above |
| **Embedding cache** | hash of text + embedding model | embedding calls when re-indexing | no | very high on re-runs |
| **Tool-result cache** | tool name + arguments | slow or rate-limited tool calls | can be stale | depends on the tool |

The rest of this notebook builds the first two, measures the danger in the second, and does the arithmetic for
the third.

## 2. Exact-match cache: the key is the design

**In plain English:** remember the answer to a question, keyed by everything that could change that answer. Two
decisions matter: **normalise** the text so trivial differences ("Hours??" vs "hours") collapse to one key, and
**include** model, prompt version, sampling parameters and tenant so a change to any of them is a miss.

In [ ]:
PROMPT_VERSION = "faq-v3"

def normalise(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^\w\s$%@.-]", " ", text)          # drop most punctuation, keep $, %, @ and decimals
    text = re.sub(r"(?<!\d)\.|\.(?!\d)", " ", text)     # a dot only survives inside a number
    return re.sub(r"\s+", " ", text).strip()

def cache_key(messages, *, model, prompt_version, tenant, **params) -> str:
    payload = {"model": model, "prompt_version": prompt_version, "tenant": tenant,
               "params": dict(sorted(params.items())),
               "messages": [{"role": m["role"], "content": normalise(m["content"])} for m in messages]}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

for q in ["What are your opening hours?", "  what are your OPENING hours??", "What are your opening hours"]:
    print(f"{q!r:<36} -> {normalise(q)!r}")
print("price with decimals survives:", normalise("Is it $49.99?"))
assert len({normalise(q) for q in ["Hours?", "hours", " HOURS!! "]}) == 1

In [ ]:
class ExactCache:
    def __init__(self, ttl_s=3600.0, clock=time.monotonic):
        self.store, self.ttl, self.clock = {}, ttl_s, clock
        self.hits = self.misses = 0
    def get(self, key):
        item = self.store.get(key)
        if item is not None and self.clock() - item[1] < self.ttl:
            self.hits += 1
            return item[0]
        self.misses += 1
        return None
    def set(self, key, value):
        self.store[key] = (value, self.clock())

def cached_chat(question, cache, *, tenant="acme", temperature=0.0, max_tokens=60):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    key = cache_key(messages, model=MODEL, prompt_version=PROMPT_VERSION, tenant=tenant,
                    temperature=temperature, max_tokens=max_tokens)
    answer = cache.get(key)
    if answer is not None:
        return answer, True
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens)
    answer = r.choices[0].message.content
    if r.choices[0].finish_reason == "stop":          # never cache a truncated, failed or degraded answer
        cache.set(key, answer)
    return answer, False

Simulated traffic: 40 requests over 8 FAQ questions with realistic popularity (a few questions dominate) and
realistic surface noise (case, spacing, extra punctuation). First the best hit rate each key allows, then the real
run through the normalised cache.

In [ ]:
rng = random.Random(0)
BASE = ["What are your opening hours?", "How long do refunds take?", "How do I reset my password?",
        "Do you ship to Canada?", "Is there a student discount?", "How do I cancel my subscription?",
        "How do I contact support?", "What payment methods do you accept?"]
weights = [1 / (i + 1) for i in range(len(BASE))]               # Zipf-like popularity
noise = [str.lower, str.upper, lambda q: f"  {q} ", lambda q: q.rstrip("?") + "??", lambda q: q, lambda q: q]
traffic = [rng.choice(noise)(rng.choices(BASE, weights)[0]) for _ in range(40)]

n_raw, n_norm = len(set(traffic)), len({normalise(q) for q in traffic})
print(f"distinct keys: raw text {n_raw}, normalised {n_norm}  ->  best possible hit rate: "
      f"raw {1 - n_raw / 40:.0%}, normalised {1 - n_norm / 40:.0%}")

norm_cache, norm_log, before = ExactCache(), [], served()
for q in traffic:
    t0 = time.perf_counter()
    answer, hit = cached_chat(q, norm_cache)
    norm_log.append((time.perf_counter() - t0, hit))
print(f"normalised cache: {norm_cache.hits} hits, {norm_cache.misses} misses -> hit rate {norm_cache.hits / 40:.0%}")
print("sample answer:", cached_chat("what are your opening hours", norm_cache)[0])
assert n_norm < n_raw and norm_cache.misses == n_norm
if OFFLINE:
    assert served() - before == n_norm, "every miss, and only a miss, reaches the provider"

In [ ]:
lat = np.array([t for t, _ in norm_log]); hits = np.array([h for _, h in norm_log])
miss_mean = lat[~hits].mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].bar(range(40), 1000 * lat, color=np.where(hits, "#2f9e44", "#adb5bd"))
axes[0].set_yscale("log")
axes[0].set(xlabel="request #", ylabel="latency (ms, log scale)", title="Grey = miss (calls the model), green = hit")
axes[1].plot(np.cumsum(np.full(40, miss_mean)), label="no cache (every request pays)", color="#adb5bd", lw=2)
axes[1].plot(np.cumsum(lat), label=f"exact cache, hit rate {hits.mean():.0%}", color="#2f9e44", lw=2)
axes[1].set(xlabel="request #", ylabel="cumulative wait (s)", title="Time saved by the cache")
axes[1].legend(); plt.tight_layout(); plt.show()
print(f"mean latency: miss {1000 * miss_mean:.0f} ms, hit {1000 * lat[hits].mean():.2f} ms")
assert lat[hits].max() < lat[~hits].min(), "every hit is faster than every miss"

**What goes in the key** is the real design question. Anything that changes the answer must change the key,
otherwise a prompt deploy "does not work" because yesterday's answers keep coming back (`po09`):

In [ ]:
cache = ExactCache()
cached_chat("What are your opening hours?", cache)                  # warm the cache
variants = {
    "same question, different case": dict(question="WHAT ARE YOUR OPENING HOURS"),
    "different tenant": dict(question="What are your opening hours?", tenant="globex"),
    "different temperature": dict(question="What are your opening hours?", temperature=0.7),
    "different max_tokens": dict(question="What are your opening hours?", max_tokens=200),
}
for label, kw in variants.items():
    _, hit = cached_chat(cache=cache, **kw)
    print(f"{label:<32} -> {'HIT' if hit else 'miss'}")
PROMPT_VERSION = "faq-v4"                                             # a prompt deploy
print(f"{'after a prompt-version bump':<32} -> {'HIT' if cached_chat('What are your opening hours?', cache)[1] else 'miss'}")
PROMPT_VERSION = "faq-v3"

Two gotchas. Caching a `temperature > 0` answer makes that answer permanent, which is usually what you want for
FAQ traffic but not for "write me a poem". And never cache anything personalised or permission-scoped without the
user or tenant in the key: a shared cache key is the most common cross-tenant data leak (`sf08`).

## 3. Semantic cache: reuse answers for *similar* questions

Exact matching misses "When will I get my money back?" versus "How long do refunds take?". A semantic cache embeds
the incoming question, finds the nearest previously answered question, and reuses its answer if the cosine
similarity clears a threshold. We use a real sentence-embedding model, `all-MiniLM-L6-v2`, so the similarities
below are what a production model gives, not the offline stand-in's word overlap.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    def embed_texts(texts):
        return st_model.encode(list(texts), normalize_embeddings=True)
    EMB_NAME = "all-MiniLM-L6-v2"
except Exception as e:                                   # no cached model and no network
    print("Could not load sentence-transformers/all-MiniLM-L6-v2 (a one-time 90 MB download):", e)
    print("Falling back to the provider's embeddings via llm_setup.embed(); the thresholds below will differ.")
    def embed_texts(texts):
        v = np.array(embed(list(texts)))
        return v / np.linalg.norm(v, axis=1, keepdims=True)
    EMB_NAME = EMBED_MODEL
print("embedding model:", EMB_NAME, "| dims:", embed_texts(["hello"]).shape[1])

In [ ]:
class SemanticCache:
    def __init__(self, threshold: float):
        self.threshold, self.entries, self.vecs = threshold, [], None
    def add(self, query, answer, **scope):
        v = embed_texts([query])
        self.vecs = v if self.vecs is None else np.vstack([self.vecs, v])
        self.entries.append({"query": query, "answer": answer, "scope": scope})
    def lookup(self, query, **scope):
        """Nearest cached question *within the same scope* (tenant, prompt version...)."""
        if not self.entries:
            return None, 0.0
        sims = self.vecs @ embed_texts([query])[0]
        in_scope = np.array([e["scope"] == scope for e in self.entries])
        sims = np.where(in_scope, sims, -1.0)
        i = int(np.argmax(sims))
        return (self.entries[i] if sims[i] >= self.threshold else None), float(sims[i])

for threshold in (0.60, 0.90):
    sc = SemanticCache(threshold)
    sc.add("What is the capital of France?", "Paris is the capital of France.", tenant="acme")
    print(f"threshold {threshold}:")
    for q in ["Which city is the capital of France?", "What is the capital of Germany?"]:
        hit, sim = sc.lookup(q, tenant="acme")
        print(f"   {q:<38} sim={sim:.3f} -> {hit['answer'] if hit else 'miss: call the model'}")

At 0.60 the cache answers "What is the capital of **Germany**?" with "Paris". That is the semantic cache's failure
mode: a confident, fast, wrong answer. Raising the threshold fixes this pair, so is the answer just "set it high"?
Measure it on a **labelled** set: pairs of questions that should share an answer (paraphrases) and pairs that look
alike but must not (hard negatives: different entity, number, direction or a negation).

In [ ]:
PARAPHRASES = [
    ("What is the capital of France?", "Which city is the capital of France?"),
    ("How do I reset my password?", "I forgot my password, how can I change it?"),
    ("How long do refunds take?", "When will I get my money back?"),
    ("Can I cancel my subscription?", "How do I cancel my plan?"),
    ("What are your opening hours?", "When are you open?"),
    ("Do you ship to Canada?", "Can you deliver to Canada?"),
    ("How do I contact support?", "What's the best way to reach customer service?"),
    ("Is there a student discount?", "Do students get a discount?"),
    ("How do I change my email address?", "Can I update the email on my account?"),
    ("What payment methods do you accept?", "Which ways can I pay?"),
    ("Where is my order?", "How can I track my package?"),
    ("How do I delete my account?", "I want to close my account permanently"),
    ("How much does shipping cost?", "What are the delivery fees?"),
    ("Can I return an item without a receipt?", "Is a receipt required to return something?"),
    ("How do I turn on two-factor authentication?", "How can I enable 2FA on my account?"),
    ("What time does the store close today?", "Until when are you open today?"),
]
HARD_NEGATIVES = [
    ("What is the capital of France?", "What is the capital of Germany?"),
    ("Can I get a refund within 30 days?", "Can I get a refund after 30 days?"),
    ("Is the premium plan refundable?", "Is the premium plan not refundable?"),
    ("How do I reset my password?", "How do I reset my username?"),
    ("Do you ship to Canada?", "Do you ship to Mexico?"),
    ("How do I cancel my subscription?", "How do I upgrade my subscription?"),
    ("What are your opening hours on Monday?", "What are your opening hours on Sunday?"),
    ("How much does the basic plan cost?", "How much does the enterprise plan cost?"),
    ("How do I add a user to my team?", "How do I remove a user from my team?"),
    ("Is my data encrypted at rest?", "Is my data encrypted in transit?"),
    ("What is the weather in Paris today?", "What is the weather in Paris tomorrow?"),
    ("Convert 10 dollars to euros", "Convert 10 euros to dollars"),
    ("How do I turn on two-factor authentication?", "How do I turn off two-factor authentication?"),
    ("Can I return an item without a receipt?", "Can I return an item with a receipt?"),
    ("What is the price of the monthly plan?", "What is the price of the annual plan?"),
    ("Do you ship to Canada?", "Do you ship from Canada?"),
]
def pair_sims(pairs):
    a, b = embed_texts([p[0] for p in pairs]), embed_texts([p[1] for p in pairs])
    return np.sum(a * b, axis=1)
sim_pos, sim_neg = pair_sims(PARAPHRASES), pair_sims(HARD_NEGATIVES)
print(f"paraphrases   : median sim {np.median(sim_pos):.3f}  (min {sim_pos.min():.3f}, max {sim_pos.max():.3f})")
print(f"hard negatives: median sim {np.median(sim_neg):.3f}  (min {sim_neg.min():.3f}, max {sim_neg.max():.3f})")
for (a, b), s in sorted(zip(HARD_NEGATIVES, sim_neg), key=lambda x: -x[1])[:4]:
    print(f"   {s:.3f}  {a!r} vs {b!r}")

In [ ]:
thresholds = np.linspace(0.3, 1.0, 141)
correct_hit = np.array([(sim_pos >= t).mean() for t in thresholds])     # paraphrase served from cache: good
false_hit = np.array([(sim_neg >= t).mean() for t in thresholds])       # different question served: wrong answer

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(sim_pos, np.full(len(sim_pos), 1) + np.random.default_rng(0).uniform(-0.08, 0.08, len(sim_pos)),
                color="#2f9e44", label="paraphrase (should hit)")
axes[0].scatter(sim_neg, np.full(len(sim_neg), 0) + np.random.default_rng(1).uniform(-0.08, 0.08, len(sim_neg)),
                color="#e03131", marker="x", label="hard negative (must miss)")
axes[0].set_yticks([0, 1], ["hard\nnegative", "paraphrase"]); axes[0].set_ylim(-0.5, 1.5)
axes[0].set(xlabel=f"cosine similarity ({EMB_NAME})", title="The two classes overlap")
axes[0].legend(loc="lower left", fontsize=8)
axes[1].plot(thresholds, correct_hit, color="#2f9e44", lw=2, label="correct-hit rate (paraphrases served)")
axes[1].plot(thresholds, false_hit, color="#e03131", lw=2, label="false-hit rate (wrong answer served)")
axes[1].set(xlabel="similarity threshold", ylabel="share of labelled pairs", title="No threshold gives hits without wrong answers")
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

useful = thresholds[correct_hit >= 0.5]
print(f"thresholds that keep >= 50% of paraphrase hits: <= {useful.max():.2f}; false-hit rate there: "
      f"{false_hit[correct_hit >= 0.5].min():.0%} at best")
assert false_hit[correct_hit >= 0.5].min() > 0, "similarity is not equivalence"

The hard negatives are not rare corner cases: "refund **within** 30 days" vs "**after** 30 days", "refundable" vs
"**not** refundable", "dollars to euros" vs "euros to dollars" sit *above* most true paraphrases, because the
embedding measures shared wording and topic, not intent. So:

- **Tune the threshold on labelled pairs from your own traffic**, per intent if you can, and re-tune when you change
  the embedding model (thresholds do not transfer between models).
- Report the **false-hit rate next to the hit rate**. A hit rate with no precision number is not a result.
- Add a cheap **verifier** on each candidate hit. Below is a generic rule-based one (numbers, negation and named
  entities must match). Production systems use a cross-encoder trained on duplicate-question data or a small LLM
  judge on the pair, because rules only catch the easy cases.
- Exclude personalised, time-sensitive and permission-scoped questions entirely, and scope entries by tenant and
  prompt version (the `scope` argument above).

In [ ]:
NEGATIONS = {"not", "no", "never", "without", "cannot", "can't", "don't", "isn't", "won't"}

def slots(text):
    words = re.findall(r"[a-z']+|\d+(?:\.\d+)?", text.lower())
    return {"numbers": {w for w in words if w[0].isdigit()},
            "negated": bool(NEGATIONS & set(words)),
            "entities": set(re.findall(r"(?<!^)\b[A-Z][a-z]+", text))}   # capitalised words after the first

def guard_ok(cached_q, new_q) -> bool:
    return slots(cached_q) == slots(new_q)

T = 0.60
guard_pos = np.array([s >= T and guard_ok(a, b) for (a, b), s in zip(PARAPHRASES, sim_pos)])
guard_neg = np.array([s >= T and guard_ok(a, b) for (a, b), s in zip(HARD_NEGATIVES, sim_neg)])
print(f"threshold {T} alone : correct hits {(sim_pos >= T).mean():.0%}, false hits {(sim_neg >= T).mean():.0%}")
print(f"threshold + guard  : correct hits {guard_pos.mean():.0%}, false hits {guard_neg.mean():.0%}")
print("still served wrongly:", [f"{a} | {b}" for (a, b), g in zip(HARD_NEGATIVES, guard_neg) if g][:4], "...")
assert 0 < guard_neg.mean() < (sim_neg >= T).mean(), "the guard helps but does not solve it"

The rules catch the "different entity, number or negation" cases (Germany, Mexico, Sunday, "not refundable",
"with/without a receipt") and cost a couple of good hits. They cannot tell "within" from "after 30 days", "add" from
"remove" or "dollars to euros" from "euros to dollars": that needs a model that reads both questions together. The
shape is what to remember: a fast recall step (embedding similarity) plus a precise verification step, both measured
on labelled pairs. The common advice of a cosine threshold of 0.95+ exists for exactly the reason in the chart, and
even that does not remove the worst hard negatives here.

## 4. Freshness: TTLs, versioned keys, stampedes

A cache that never forgets serves yesterday's prices. Three mechanisms, usually combined (`in42`):

1. **TTL by intent class**: short for volatile answers (prices, stock, status), long for stable ones (policies).
2. **Versioned keys**: put the ids *and versions* of the retrieved documents into the key. Edit the refund policy
   (v3 to v4) and every answer built on it simply stops matching. No purge job at all.
3. **Event-driven purge**: the ingestion pipeline says "doc 42 changed" and you evict keys that reference it (needs
   a reverse index from document to keys).

In [ ]:
class FakeClock:
    def __init__(self): self.t = 0.0
    def __call__(self): return self.t

clock = FakeClock()
TTL_BY_INTENT = {"pricing": 60.0, "policy": 86_400.0}
caches = {intent: ExactCache(ttl_s=ttl, clock=clock) for intent, ttl in TTL_BY_INTENT.items()}
caches["pricing"].set("price-basic", "The basic plan is $10/month.")
caches["policy"].set("refund-policy", "Refunds within 30 days.")
clock.t = 120.0                                                        # two minutes later
print("pricing answer after 2 min:", caches["pricing"].get("price-basic"))
print("policy answer after 2 min :", caches["policy"].get("refund-policy"))

def rag_cache_key(question, retrieved):                 # retrieved = [(doc_id, version), ...]
    return cache_key([{"role": "user", "content": question}], model=MODEL, prompt_version=PROMPT_VERSION,
                     tenant="acme", sources=sorted(retrieved))
k_v3 = rag_cache_key("Can I get a refund?", [("refund-policy", 3), ("faq", 12)])
k_v4 = rag_cache_key("Can I get a refund?", [("refund-policy", 4), ("faq", 12)])
print("key changes when the refund policy goes v3 -> v4:", k_v3 != k_v4)
assert caches["pricing"].get("price-basic") is None and k_v3 != k_v4

**Stampede vs penetration** (`in41`). A *stampede* is one hot key expiring while hundreds of requests want it: they
all miss at once and all call the model. The fix is **single-flight**: the first miss computes, everyone else awaits
that same result. (*Penetration* is the other problem: requests for keys that can never be cached, such as a random
string in every question. Fix it with negative caching, key validation and an alarm on per-caller miss rate.)

In [ ]:
class SingleFlight:
    def __init__(self):
        self.inflight = {}
    async def do(self, key, fn):
        task = self.inflight.get(key)
        if task is None:                                             # first caller for this key: start the work
            task = asyncio.ensure_future(fn())
            self.inflight[key] = task
            task.add_done_callback(lambda _: self.inflight.pop(key, None))
        return await task                                            # everyone awaits the same task

backend_calls = 0
async def expensive_llm_answer():
    global backend_calls
    backend_calls += 1
    await asyncio.sleep(0.2)                                         # a fake, free model call
    return "Our opening hours are 9am to 6pm."

await asyncio.gather(*(expensive_llm_answer() for _ in range(50)))
without = backend_calls
backend_calls, sf = 0, SingleFlight()
answers = await asyncio.gather(*(sf.do("hours", expensive_llm_answer) for _ in range(50)))
print(f"50 simultaneous misses on a cold key: {without} model calls without single-flight, {backend_calls} with it")
assert backend_calls == 1 and len(set(answers)) == 1

## 5. Provider prompt caching (prefix caching)

This one is the provider's cache, and for most systems it is the **largest single cost lever**: the provider keeps
the processed (KV) state of a prompt *prefix* it has seen recently and, on a match, skips that part of prefill. You
pay less for the cached input tokens and the first token arrives sooner.

How OpenAI does it (as documented at the time of writing; check the current docs): automatic, no code change, for
prompts of **1,024 tokens or more**; matches the longest previously seen prefix in **128-token steps**; cached
entries are evicted after minutes of inactivity; the response reports `usage.prompt_tokens_details.cached_tokens`.
The discount depends on the model: for example `gpt-4.1-mini` is listed at $0.40 per 1M input tokens and $0.10 per 1M
cached input tokens (75% off); other families have had 50% or 90%. Anthropic needs explicit `cache_control`
breakpoints; Gemini has implicit and explicit context caching.

The code to read it is one line. The stand-in always reports 0; with an OpenAI key the second call below should
report most of the long prefix as cached.

In [ ]:
enc = tiktoken.get_encoding("o200k_base")                 # the GPT-4o / GPT-4.1 tokenizer
POLICY = "\n".join(f"Policy {i}: requests of type {i} are handled by team {i % 7} within {i % 5 + 1} business days, "
                   f"unless the customer is on the enterprise plan, in which case escalation goes to the account manager."
                   for i in range(1, 41))
LONG_SYSTEM = SYSTEM + "\n\nPOLICY HANDBOOK\n" + POLICY
print(f"stable prefix: {len(enc.encode(LONG_SYSTEM)):,} tokens (needs >= 1,024 to be cacheable)")
for i in range(2):
    r = client.chat.completions.create(model=MODEL, max_tokens=30, messages=[
        {"role": "system", "content": LONG_SYSTEM}, {"role": "user", "content": "What are your opening hours?"}])
    details = r.usage.prompt_tokens_details
    print(f"call {i + 1}: prompt_tokens={r.usage.prompt_tokens:,}  cached_tokens={details.cached_tokens if details else 0}")

**Layout decides the hit rate.** The cache is prefix-only: one changed token near the top invalidates everything
after it. The classic bug is a timestamp or request id at the top of the system prompt (`pd19`). We simulate five
consecutive requests with two layouts, counting cacheable tokens with OpenAI's rules above.

In [ ]:
def build_prompt(question, now, layout):
    stable = LONG_SYSTEM
    if layout == "stable first":
        return f"{stable}\nCurrent time: {now}\nUser question: {question}"
    return f"Current time: {now}\n{stable}\nUser question: {question}"        # the bug

def cached_prefix_tokens(prev_ids, ids, minimum=1024, step=128):
    n = next((i for i, (a, b) in enumerate(zip(prev_ids, ids)) if a != b), min(len(prev_ids), len(ids)))
    return 0 if n < minimum else (n // step) * step

questions = ["What are your opening hours?", "How long do refunds take?", "Do you ship to Canada?",
             "How do I contact support?", "Is there a student discount?"]
results = {}
for layout in ("stable first", "timestamp first"):
    prev, rows = [], []
    for k, q in enumerate(questions):
        ids = enc.encode(build_prompt(q, f"2026-09-29 10:00:{k:02d}", layout))
        rows.append((len(ids), cached_prefix_tokens(prev, ids)))
        prev = ids
    results[layout] = rows
    print(f"{layout:<16}: cached tokens per request = {[c for _, c in rows]}")
assert all(c > 1000 for _, c in results["stable first"][1:]) and all(c == 0 for _, c in results["timestamp first"])

In [ ]:
PRICE_IN, PRICE_CACHED, PRICE_OUT = 0.40, 0.10, 1.60        # $ per 1M tokens, gpt-4.1-mini list prices (check yours)
def request_cost(prompt_tokens, cached, out_tokens=60):
    return ((prompt_tokens - cached) * PRICE_IN + cached * PRICE_CACHED + out_tokens * PRICE_OUT) / 1e6

fig, ax = plt.subplots(figsize=(9, 3.6))
for j, (layout, rows) in enumerate(results.items()):
    x = np.arange(len(rows)) + j * (len(rows) + 1.5)             # two separate groups of five requests
    cached = np.array([c for _, c in rows]); total = np.array([n for n, _ in rows])
    ax.bar(x, total - cached, width=0.8, color="#adb5bd", label="uncached input tokens" if j == 0 else None)
    ax.bar(x, cached, bottom=total - cached, width=0.8, color="#2f9e44", label="cached input tokens" if j == 0 else None)
    cost = np.mean([request_cost(n, c) for n, c in rows[1:]])
    ax.text(x.mean(), total.max() * 1.04, f"{layout}\n${cost * 1e6:,.0f} per 1M requests", ha="center", fontsize=9)
ax.set_xticks([]); ax.set_xlabel("five consecutive requests per layout")
ax.set_ylim(0, 1.35 * max(n for rows in results.values() for n, _ in rows))
ax.set(ylabel="input tokens per request", title="Same prompt, two layouts: the timestamp at the top kills the cache")
ax.legend(loc="lower right"); plt.show()

Rules that keep it working: stable content first (system prompt, tool schemas in a fixed order, few-shot block,
the document), volatile content last (the question, timestamps, ids, per-turn retrieved chunks); batch prompt edits
(each one invalidates the cache fleet-wide, so a canary looks expensive until it warms); alarm on the cached-token
ratio (`usage.prompt_tokens_details.cached_tokens / prompt_tokens`).

## 6. KV and prefix caching inside your own serving engine

The same idea on your own GPUs. Inside one generation the **KV cache** stores every past token's keys and values so
decode does not recompute them (see [KV cache and inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb)).
**Prefix caching** keeps those KV blocks *across requests*, so a shared system prompt is prefilled once.

| | KV cache | Prefix / prompt cache |
|---|---|---|
| Lives | inside one request | across requests |
| Saves | recomputing past tokens at every decode step | prefill of a shared prefix |
| Helps | tokens per second (decode) | time to first token and input cost |
| Where | every transformer inference engine | provider-side (OpenAI, Anthropic, Gemini), vLLM automatic prefix caching, SGLang RadixAttention |

With vLLM (needs a GPU and `pip install vllm`; not runnable here), prefix caching is a flag, and recent versions
turn it on by default:

```python
from vllm import LLM, SamplingParams
llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct", enable_prefix_caching=True)
outs = llm.generate([LONG_SYSTEM + "\nQ: opening hours?", LONG_SYSTEM + "\nQ: refunds?"], SamplingParams(max_tokens=50))
```

## 7. Embedding cache

Re-indexing a corpus re-embeds every chunk, most of which did not change. Key on the text **and the model name**
(a new model must re-embed everything), deduplicate within a batch, and send only the misses in one batched call.

In [ ]:
class EmbeddingCache:
    def __init__(self, model: str):
        self.model, self.store, self.texts_sent, self.api_calls = model, {}, 0, 0
    def _key(self, text):
        return hashlib.sha256(f"{self.model}\x00{text}".encode()).hexdigest()
    def embed(self, texts):
        keys = [self._key(t) for t in texts]
        todo = list(dict.fromkeys(t for t, k in zip(texts, keys) if k not in self.store))   # misses, deduplicated
        if todo:
            for t, v in zip(todo, embed(todo, model=self.model)):                          # one batched API call
                self.store[self._key(t)] = v
            self.texts_sent += len(todo); self.api_calls += 1
        return [self.store[k] for k in keys]

corpus = [f"Chunk {i}: section {i} of the product manual." for i in range(25)] + ["Copyright Acme Inc."] * 5
ec = EmbeddingCache(EMBED_MODEL)
ec.embed(corpus)
first = ec.texts_sent
corpus[3], corpus[7], corpus[11] = corpus[3] + " (edited)", corpus[7] + " (edited)", corpus[11] + " (edited)"
ec.embed(corpus)
print(f"first index run : {len(corpus)} chunks -> {first} texts embedded (the 5 boilerplate copies deduplicated)")
print(f"second run      : {len(corpus)} chunks -> {ec.texts_sent - first} texts embedded (only the edits)")
assert first == 26 and ec.texts_sent - first == 3

## 8. Hit rate is not cost saving

A hit on a cheap request saves less than a hit on an expensive one (`pd18`). Two traffic mixes with the same 40%
hit rate: in one the hits land on short FAQ answers, in the other on long document summaries.

In [ ]:
REQUEST_TYPES = {"FAQ answer": (0.60, 0.0004), "RAG answer": (0.30, 0.004), "document summary": (0.10, 0.04)}  # share, $/request
SCENARIOS = {"hits on cheap FAQs": {"FAQ answer": 0.60, "RAG answer": 0.133, "document summary": 0.0},
             "hits on long summaries": {"FAQ answer": 0.20, "RAG answer": 0.80, "document summary": 0.40}}

def cache_effect(hit_rates):
    hit = sum(REQUEST_TYPES[k][0] * h for k, h in hit_rates.items())
    spend = sum(share * cost for share, cost in REQUEST_TYPES.values())
    saved = sum(REQUEST_TYPES[k][0] * REQUEST_TYPES[k][1] * h for k, h in hit_rates.items())
    return hit, saved / spend, spend

daily_requests = 200_000
fig, ax = plt.subplots(figsize=(8, 3.4))
for j, (name, rates) in enumerate(SCENARIOS.items()):
    hit, saving, spend = cache_effect(rates)
    bars = ax.bar([j - 0.18, j + 0.18], [hit, saving], width=0.34, color=["#74c0fc", "#2f9e44"],
                  label=["hit rate (by count)", "cost saved (by $)"] if j == 0 else None)
    ax.bar_label(bars, labels=[f"{hit:.0%}", f"{saving:.0%}"])
    print(f"{name:<24}: hit rate {hit:.0%}, cost saved {saving:.0%}  -> "
          f"${daily_requests * spend * saving * 30:,.0f} saved per month at {daily_requests:,} requests/day")
ax.set_xticks([0, 1], list(SCENARIOS)); ax.set_ylim(0, 0.85)
ax.set(ylabel="share", title="Same hit rate, very different savings: weight cache metrics by cost")
ax.legend(); plt.show()

Report savings per cache layer, weighted by cost, and net of what the cache itself costs (Redis, a vector index,
engineering time). It changes which optimisation you do next.

## Try it yourself

**1.** You are allowed a false-hit rate of at most 5% on the labelled set. Which threshold gives the most correct
hits under that constraint, and how many paraphrases does it still catch?

In [ ]:
# Solution — try it yourself first, then run this
ok = false_hit <= 0.05
best = thresholds[ok][np.argmax(correct_hit[ok])]
print(f"best threshold under a 5% false-hit budget: {best:.3f} -> correct-hit rate {correct_hit[thresholds == best][0]:.0%}")
print("On these hard negatives, similarity alone cannot meet the budget and still hit: you need the verifier step,")
print("per-intent thresholds, or to keep the semantic cache away from questions where details flip the answer.")

**2.** Prove the tenant scope works: cache an answer for tenant `acme`, then look up the identical question as
tenant `globex`. It must miss.

In [ ]:
# Solution — try it yourself first, then run this
sc = SemanticCache(threshold=0.9)
sc.add("What is our refund window?", "Acme: 30 days.", tenant="acme", prompt_version="faq-v3")
same_tenant, s1 = sc.lookup("What is our refund window?", tenant="acme", prompt_version="faq-v3")
other_tenant, s2 = sc.lookup("What is our refund window?", tenant="globex", prompt_version="faq-v3")
print("acme  ->", same_tenant["answer"] if same_tenant else None)
print("globex->", other_tenant["answer"] if other_tenant else "miss (correct: never serve another tenant's answer)")
assert same_tenant and other_tenant is None

**3.** A conversation of 20 turns re-sends a 1,200-token stable prefix every turn. What does prompt caching save
per conversation at the prices above, if every turn after the first gets 1,152 cached tokens?

In [ ]:
# Solution — try it yourself first, then run this
turns, prefix, cached_t = 20, 1200, 1152
without = turns * prefix * PRICE_IN / 1e6
with_cache = (prefix * PRICE_IN + (turns - 1) * ((prefix - cached_t) * PRICE_IN + cached_t * PRICE_CACHED)) / 1e6
print(f"prefix cost per conversation: ${without:.5f} without caching, ${with_cache:.5f} with -> {1 - with_cache / without:.0%} saved")

if OFFLINE:                          # reset the stand-in's knobs
    OFFLINE.latency = 0.0

## Say it in an interview

- **30 seconds:** "I layer caches. Normalise and hash for an exact cache keyed on the question, model, parameters,
  prompt version, tenant and the versions of the retrieved documents, so invalidation is automatic. Keep the prompt
  prefix byte-stable so the provider's prompt cache hits. Then, only for FAQ-shaped traffic, a semantic cache with a
  threshold tuned on labelled paraphrase pairs from our own logs, a verifier on each hit, and a false-hit rate I
  report next to the hit rate."
- **Numbers:** exact-cache hits cost well under a millisecond here vs ~300 ms for a call; OpenAI caches prefixes of
  1,024+ tokens in 128-token steps; cached input is discounted (75% for `gpt-4.1-mini` at list price, other models
  differ); semantic hit rates of 20-50% are realistic only on repetitive traffic.
- **The danger:** similarity is not equivalence. In this notebook's labelled set the hard negatives ("within" vs
  "after" 30 days, "refundable" vs "not refundable") scored *higher* than most true paraphrases.
- **Follow-ups:** stampede (single-flight, serve-stale-while-revalidate, TTL jitter), penetration (negative caching,
  a miss-rate alarm), tenant isolation (tenant in every key), never caching degraded or truncated answers.
- **Traps:** a timestamp at the top of the system prompt; forgetting the prompt version in the key; one global TTL;
  quoting hit rate instead of cost saved; "0.85 is fine".

## Next

- [04 · FastAPI serving and streaming](04_fastapi_serving_and_streaming.ipynb): where the cache sits in the request path.
- [05 · Latency percentiles and cost](05_latency_percentiles_p50_p95_and_cost.ipynb): the full cost calculator.
- [06 · LiteLLM gateway and model routing](06_litellm_gateway_and_model_routing.ipynb): the other big cost lever.
- Related: [embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb) · [KV cache](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb) · [RAG pipeline](../09_rag/06_rag_pipeline_from_scratch.ipynb).
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) (tune a semantic-cache threshold live) · [interview bank](../../ai_interview_prep/index.html).